# CO3133 - Text Classification with Jigsaw Toxic Comment

This notebook implements the full **text classification** pipeline for BTL1.

## Dataset description
- **Dataset:** Jigsaw Toxic Comment Classification Challenge
- **Task:** multi-label classification with 6 target labels
- **Labels:** `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate`
- **Input:** one user comment
- **Output:** a 6-dimensional multi-hot prediction vector

## Notebook structure
1. Resolve project paths from the BTL1 folder
2. Configure dataset paths and training hyperparameters
3. Extract and validate the raw Kaggle archive
4. Prepare train/validation/test splits and run EDA
5. Fine-tune **BERT**
6. Train an **LSTM** baseline
7. Export metrics, plots, and error-analysis artifacts for the report website

The extraction step is safe to re-run: it becomes a no-op when the expected CSV files already exist.


In [ ]:
from pathlib import Path
import json
import math
import random
import re
import warnings
import zipfile
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")


def find_btl1_root() -> Path:
    # Resolve the BTL1 folder from the current working directory.
    # This keeps the notebook independent from any unrelated root marker file.
    for base in [Path.cwd(), *Path.cwd().parents]:
        if base.name == "btl1" and (base / "data").exists() and (base / "artifacts").exists():
            return base
        candidate = base / "btl1"
        if candidate.exists() and (candidate / "data").exists() and (candidate / "artifacts").exists():
            return candidate
    raise FileNotFoundError("Could not locate the btl1 directory. Run the notebook from the repo root or from inside btl1/.")


BTL1_ROOT = find_btl1_root()
REPO_ROOT = BTL1_ROOT.parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"BTL1 root: {BTL1_ROOT}")
print(f"Device: {DEVICE}")


In [ ]:
TEXT_DIR = BTL1_ROOT / "data" / "text" / "jigsaw"
SOURCE_DIR = TEXT_DIR / "source"
RAW_DIR = TEXT_DIR / "raw"
PROCESSED_DIR = TEXT_DIR / "processed"
ARTIFACT_DIR = BTL1_ROOT / "artifacts" / "text"

ZIP_PATH = SOURCE_DIR / "jigsaw-toxic-comment-classification-challenge.zip"
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

SEED = 42
TEST_RATIO = 0.10
VAL_RATIO = 0.10
MAX_ROWS = None  # Set an integer for quick debugging if needed.

for folder in [RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Source zip: {ZIP_PATH}")
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Artifact dir: {ARTIFACT_DIR}")


In [ ]:
BERT_CFG = {
    "model_name": "bert-base-uncased",
    "max_length": 192,
    "batch_size": 16,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "epochs": 5,
}

LSTM_CFG = {
    "batch_size": 64,
    "embed_dim": 200,
    "hidden_dim": 128,
    "dropout": 0.3,
    "lr": 1e-3,
    "epochs": 5,
    "max_length": 300,
    "max_vocab": 50000,
    "min_freq": 2,
}

PATIENCE = 2

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
def extract_zip(archive_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "r") as handle:
        handle.extractall(destination)


def extract_jigsaw_dataset(zip_path: Path, raw_dir: Path):
    # Kaggle ships one outer archive with several nested archives inside.
    # We only extract when the expected CSV files are missing.
    train_candidates = list(raw_dir.rglob("train.csv"))
    test_candidates = list(raw_dir.rglob("test.csv"))
    labels_candidates = list(raw_dir.rglob("test_labels.csv"))
    if train_candidates and test_candidates and labels_candidates:
        print("Raw Jigsaw files already exist. Skipping extraction.")
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Missing source archive: {zip_path}")

    stage_dir = raw_dir / "stage_1"
    extract_zip(zip_path, stage_dir)

    nested_archives = sorted(stage_dir.glob("*.zip"))
    if not nested_archives:
        raise FileNotFoundError("No nested archives were found inside the Kaggle download.")

    for nested_zip in nested_archives:
        target_dir = raw_dir / nested_zip.stem
        if list(target_dir.rglob("*.csv")):
            continue
        extract_zip(nested_zip, target_dir)


def find_one(root: Path, name: str) -> Path:
    matches = sorted(path for path in root.rglob(name) if path.is_file())
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {root}")
    return matches[0]


def get_csv_paths(raw_dir: Path) -> dict[str, Path]:
    return {
        "train": find_one(raw_dir, "train.csv"),
        "test": find_one(raw_dir, "test.csv"),
        "test_labels": find_one(raw_dir, "test_labels.csv"),
        "sample_submission": find_one(raw_dir, "sample_submission.csv"),
    }


def normalize_text(text: str) -> str:
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text).strip()
    return text
    return text


def exact_match_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.all(y_true == y_pred, axis=1).mean())


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, loss_value: float) -> dict:
    return {
        "exact_match_accuracy": exact_match_score(y_true, y_pred),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "loss": float(loss_value),
    }


def plot_history(history_df: pd.DataFrame, metric_col: str, title: str, output_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df[metric_col], marker="o", color="#17365d")
    axes[1].set_title(metric_col)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel(metric_col)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def build_error_payload(y_true: np.ndarray, y_pred: np.ndarray, labels: list[str]) -> dict:
    payload = {
        "exact_match_failures": int((~np.all(y_true == y_pred, axis=1)).sum()),
        "total_examples": int(y_true.shape[0]),
        "total_true_positive_labels": int(y_true.sum()),
        "total_predicted_positive_labels": int(y_pred.sum()),
        "per_label": [],
    }

    for idx, label in enumerate(labels):
        truth = y_true[:, idx]
        pred = y_pred[:, idx]
        tp = int(((truth == 1) & (pred == 1)).sum())
        fp = int(((truth == 0) & (pred == 1)).sum())
        fn = int(((truth == 1) & (pred == 0)).sum())
        tn = int(((truth == 0) & (pred == 0)).sum())
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        payload["per_label"].append({
            "label": label,
            "support": int(truth.sum()),
            "predicted_positive": int(pred.sum()),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    payload["hardest_labels"] = [item["label"] for item in sorted(payload["per_label"], key=lambda item: item["f1"])[:3]]
    return payload


In [ ]:
extract_jigsaw_dataset(ZIP_PATH, RAW_DIR)
csv_paths = get_csv_paths(RAW_DIR)

train_df = pd.read_csv(csv_paths["train"])
if MAX_ROWS is not None:
    train_df = train_df.head(MAX_ROWS).copy()

train_df["comment_text"] = train_df["comment_text"].fillna("").map(normalize_text)
train_df["char_length"] = train_df["comment_text"].str.len()
train_df["word_length"] = train_df["comment_text"].str.split().str.len()
train_df["label_count"] = train_df[LABELS].sum(axis=1)

model_df = train_df[["id", "comment_text"] + LABELS].copy()
train_df_split, temp_df = train_test_split(model_df, test_size=TEST_RATIO + VAL_RATIO, random_state=SEED, shuffle=True)
val_share = VAL_RATIO / (TEST_RATIO + VAL_RATIO)
val_df, test_df = train_test_split(temp_df, test_size=1 - val_share, random_state=SEED, shuffle=True)

train_df_split = train_df_split.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df_split.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print({"train": len(train_df_split), "val": len(val_df), "test": len(test_df)})

label_counts = train_df[LABELS].sum().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(x=label_counts.index, y=label_counts.values, ax=axes[0], color="#17365d")
axes[0].set_title("Label distribution")
axes[0].set_ylabel("Positive examples")
axes[0].tick_params(axis="x", rotation=25)
sns.histplot(train_df["word_length"], bins=50, ax=axes[1], color="#b38b2d")
axes[1].set_title("Comment length distribution")
axes[1].set_xlabel("Words per comment")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "jigsaw_eda.png", dpi=180, bbox_inches="tight")
plt.close(fig)

display(pd.DataFrame({
    "stat": ["rows", "avg chars", "median chars", "multi-label rows"],
    "value": [
        len(train_df),
        round(train_df["char_length"].mean(), 2),
        round(train_df["char_length"].median(), 2),
        int((train_df["label_count"] > 1).sum()),
    ],
}))
display(label_counts.rename("positive_examples").to_frame())


In [ ]:
class TextDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        labels = row[LABELS].astype(np.float32).to_numpy()
        return row["comment_text"], labels


tokenizer = AutoTokenizer.from_pretrained(BERT_CFG["model_name"])


def collate_batch(batch):
    texts = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch], dtype=torch.float32)
    encoded = tokenizer(texts, padding=True, truncation=True, max_length=BERT_CFG["max_length"], return_tensors="pt")
    encoded["labels"] = labels
    return encoded


train_loader = DataLoader(TextDataset(train_df_split), batch_size=BERT_CFG["batch_size"], shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(TextDataset(val_df), batch_size=BERT_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(TextDataset(test_df), batch_size=BERT_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)


def run_epoch(model, loader, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for batch in loader:
            labels = batch.pop("labels").to(DEVICE)
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            if training:
                optimizer.zero_grad()
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
            preds = (torch.sigmoid(logits) >= 0.5).int().detach().cpu().numpy()
            y_pred.append(preds)
            y_true.append(labels.detach().cpu().int().numpy())
            total_loss += float(loss.item()) * labels.size(0)

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)
    avg_loss = total_loss / len(loader.dataset)
    return compute_metrics(y_true, y_pred, avg_loss), y_true, y_pred


model = AutoModelForSequenceClassification.from_pretrained(BERT_CFG["model_name"], num_labels=len(LABELS), problem_type="multi_label_classification").to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=BERT_CFG["lr"], weight_decay=BERT_CFG["weight_decay"])
total_steps = len(train_loader) * BERT_CFG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(total_steps // 10, 1), num_training_steps=total_steps)

ckpt_path = ARTIFACT_DIR / "bert_multilabel_best.pt"
history_path = ARTIFACT_DIR / "bert_history.csv"
curve_path = ARTIFACT_DIR / "bert_learning_curves.png"

if ckpt_path.exists() and history_path.exists():
    print("Loading cached BERT checkpoint and history.")
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    history_df = pd.read_csv(history_path)
else:
    rows = []
    best_score = -math.inf
    best_state = None
    patience_left = PATIENCE
    for epoch in range(1, BERT_CFG["epochs"] + 1):
        train_metrics, _, _ = run_epoch(model, train_loader, optimizer, scheduler)
        val_metrics, _, _ = run_epoch(model, val_loader)
        rows.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
            "val_micro_f1": val_metrics["micro_f1"],
            "val_macro_f1": val_metrics["macro_f1"],
        })
        if val_metrics["micro_f1"] > best_score:
            best_score = val_metrics["micro_f1"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
    if best_state is None:
        raise RuntimeError("BERT training did not produce a valid checkpoint.")
    torch.save(best_state, ckpt_path)
    model.load_state_dict(best_state)
    history_df = pd.DataFrame(rows)
    history_df.to_csv(history_path, index=False)

plot_history(history_df, "val_micro_f1", "BERT learning curves", curve_path)
bert_metrics, bert_y_true, bert_y_pred = run_epoch(model, test_loader)
bert_metrics


In [ ]:
token_pattern = re.compile(r"[a-z']+")


def tokenize(text: str) -> list[str]:
    return token_pattern.findall(text.lower())


def build_vocab(texts, max_size=50000, min_freq=2):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq in counter.most_common():
        if freq < min_freq or len(vocab) >= max_size:
            break
        vocab[word] = len(vocab)
    return vocab


def encode_text(text: str, vocab: dict[str, int], max_length: int):
    ids = [vocab.get(token, vocab["<unk>"]) for token in tokenize(text)[:max_length]]
    return ids if ids else [vocab["<unk>"]]


class SequenceDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, vocab: dict[str, int], max_length: int):
        self.frame = frame.reset_index(drop=True)
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        token_ids = encode_text(row["comment_text"], self.vocab, self.max_length)
        labels = row[LABELS].astype(np.float32).to_numpy()
        return token_ids, labels


def collate_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(item) for item in sequences], dtype=torch.long)
    max_length = max(lengths).item()
    padded = torch.zeros(len(sequences), max_length, dtype=torch.long)
    for row_idx, token_ids in enumerate(sequences):
        padded[row_idx, : len(token_ids)] = torch.tensor(token_ids, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.float32)
    return padded, lengths, labels


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, lengths):
        embeddings = self.embedding(input_ids)
        packed = nn.utils.rnn.pack_padded_sequence(embeddings, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.encoder(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.classifier(self.dropout(pooled))


vocab = build_vocab(train_df_split["comment_text"], max_size=LSTM_CFG["max_vocab"], min_freq=LSTM_CFG["min_freq"])
(ARTIFACT_DIR / "lstm_vocab.json").write_text(json.dumps(vocab, indent=2), encoding="utf-8")

train_loader = DataLoader(SequenceDataset(train_df_split, vocab, LSTM_CFG["max_length"]), batch_size=LSTM_CFG["batch_size"], shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(SequenceDataset(val_df, vocab, LSTM_CFG["max_length"]), batch_size=LSTM_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(SequenceDataset(test_df, vocab, LSTM_CFG["max_length"]), batch_size=LSTM_CFG["batch_size"], shuffle=False, num_workers=0, collate_fn=collate_batch)


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for input_ids, lengths, labels in loader:
            input_ids = input_ids.to(DEVICE)
            lengths = lengths.to(DEVICE)
            labels = labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            preds = (torch.sigmoid(logits) >= 0.5).int().detach().cpu().numpy()
            y_pred.append(preds)
            y_true.append(labels.detach().cpu().int().numpy())
            total_loss += float(loss.item()) * labels.size(0)
    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)
    avg_loss = total_loss / len(loader.dataset)
    return compute_metrics(y_true, y_pred, avg_loss), y_true, y_pred


model = LSTMClassifier(vocab_size=len(vocab), embed_dim=LSTM_CFG["embed_dim"], hidden_dim=LSTM_CFG["hidden_dim"], num_labels=len(LABELS), dropout=LSTM_CFG["dropout"]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_CFG["lr"])

ckpt_path = ARTIFACT_DIR / "lstm_multilabel_best.pt"
history_path = ARTIFACT_DIR / "lstm_history.csv"
curve_path = ARTIFACT_DIR / "lstm_learning_curves.png"

if ckpt_path.exists() and history_path.exists():
    print("Loading cached LSTM checkpoint and history.")
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    history_df = pd.read_csv(history_path)
else:
    rows = []
    best_score = -math.inf
    best_state = None
    patience_left = PATIENCE
    for epoch in range(1, LSTM_CFG["epochs"] + 1):
        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics, _, _ = run_epoch(model, val_loader, criterion)
        rows.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
            "val_micro_f1": val_metrics["micro_f1"],
            "val_macro_f1": val_metrics["macro_f1"],
        })
        if val_metrics["micro_f1"] > best_score:
            best_score = val_metrics["micro_f1"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                break
    if best_state is None:
        raise RuntimeError("LSTM training did not produce a valid checkpoint.")
    torch.save(best_state, ckpt_path)
    model.load_state_dict(best_state)
    history_df = pd.DataFrame(rows)
    history_df.to_csv(history_path, index=False)

plot_history(history_df, "val_micro_f1", "LSTM learning curves", curve_path)
lstm_metrics, lstm_y_true, lstm_y_pred = run_epoch(model, test_loader, criterion)
lstm_metrics


In [ ]:
bert_report = classification_report(bert_y_true, bert_y_pred, target_names=LABELS, output_dict=True, zero_division=0)
lstm_report = classification_report(lstm_y_true, lstm_y_pred, target_names=LABELS, output_dict=True, zero_division=0)

comparison_df = pd.DataFrame([
    {"model": "BERT", "exact_match_accuracy": bert_metrics["exact_match_accuracy"], "micro_f1": bert_metrics["micro_f1"], "macro_f1": bert_metrics["macro_f1"]},
    {"model": "LSTM", "exact_match_accuracy": lstm_metrics["exact_match_accuracy"], "micro_f1": lstm_metrics["micro_f1"], "macro_f1": lstm_metrics["macro_f1"]},
])
comparison_df.to_csv(ARTIFACT_DIR / "text_model_comparison.csv", index=False)

per_label_df = pd.DataFrame({
    "label": LABELS,
    "bert_f1": [bert_report[label]["f1-score"] for label in LABELS],
    "lstm_f1": [lstm_report[label]["f1-score"] for label in LABELS],
})
per_label_df.to_csv(ARTIFACT_DIR / "text_per_label_f1.csv", index=False)

metrics_payload = {"bert": bert_metrics, "lstm": lstm_metrics}
(ARTIFACT_DIR / "text_metrics_summary.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

error_payload = {
    "labels": LABELS,
    "bert": build_error_payload(bert_y_true, bert_y_pred, LABELS),
    "lstm": build_error_payload(lstm_y_true, lstm_y_pred, LABELS),
}
(ARTIFACT_DIR / "text_error_analysis.json").write_text(json.dumps(error_payload, indent=2), encoding="utf-8")

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(LABELS))
width = 0.35
ax.bar(x - width / 2, per_label_df["bert_f1"], width=width, label="BERT", color="#17365d")
ax.bar(x + width / 2, per_label_df["lstm_f1"], width=width, label="LSTM", color="#b38b2d")
ax.set_xticks(x)
ax.set_xticklabels(LABELS, rotation=25)
ax.set_ylabel("F1 score")
ax.set_title("Per-label F1 on the test split")
ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "text_per_label_f1.png", dpi=180, bbox_inches="tight")
plt.close(fig)

display(comparison_df)
display(per_label_df)
